# Optimal Transport Primer

This notebook introduces the key concepts from optimal transport theory used throughout this library:
**Wasserstein distance**, **transport maps**, **displacement interpolation**, and how they
compare to classical divergences (KL, TV).


In [1]:
import sys; sys.path.insert(0, "..")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from otreturns.distributions import EmpiricalDistribution
from otreturns.distances import wasserstein_1d
from otreturns.transport import optimal_transport_map_1d, transport_cost_decomposition
from otreturns.interpolation import interpolation_path
print("Imports OK")


Imports OK


## 1. The 1-d Wasserstein Distance

$$W_p^p(\mu, \nu) = \int_0^1 |F_\mu^{-1}(u) - F_\nu^{-1}(u)|^p \, du$$

In 1-d this has an exact closed form via **quantile coupling** — the only optimal transport plan is to pair the $u$-quantile of $\mu$ with the $u$-quantile of $\nu$.

For two Gaussians: $W_2^2(\mathcal{N}(\mu_1, \sigma_1^2), \mathcal{N}(\mu_2, \sigma_2^2)) = (\mu_1-\mu_2)^2 + (\sigma_1 - \sigma_2)^2$

Note it averages **standard deviations**, not variances.


In [2]:
rng = np.random.default_rng(0)

# Two distributions: N(0,1) and N(2, 1.5^2)
mu1 = EmpiricalDistribution(rng.normal(0, 1.0, 2000))
mu2 = EmpiricalDistribution(rng.normal(2, 1.5, 2000))

w2_empirical = wasserstein_1d(mu1, mu2, p=2)
w2_theory    = np.sqrt((0 - 2)**2 + (1.0 - 1.5)**2)

print(f"W₂(μ₁, μ₂) empirical : {w2_empirical:.4f}")
print(f"W₂(μ₁, μ₂) Gaussian  : {w2_theory:.4f}  (exact formula)")

# Compare to different p values
for p in [1, 2]:
    print(f"  W_{p} = {wasserstein_1d(mu1, mu2, p=p):.4f}")


W₂(μ₁, μ₂) empirical : 2.0855
W₂(μ₁, μ₂) Gaussian  : 2.0616  (exact formula)
  W_1 = 2.0248
  W_2 = 2.0855


## 2. Quantile Functions and Transport Maps

The **optimal transport map** $T: \mathbb{R} \to \mathbb{R}$ satisfying $T\#\mu = \nu$ is:
$$T(x) = F_\nu^{-1}(F_\mu(x))$$

For Gaussians this is an **affine** map: $T(x) = \mu_\nu + \frac{\sigma_\nu}{\sigma_\mu}(x - \mu_\mu)$.


In [3]:
u = np.linspace(0.01, 0.99, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Quantile functions
axes[0].plot(u, mu1.quantile_function(u), color="steelblue", lw=2, label="μ₁ ~ N(0,1)")
axes[0].plot(u, mu2.quantile_function(u), color="darkorange", lw=2, label="μ₂ ~ N(2,1.5)")
axes[0].set_xlabel("Quantile u"); axes[0].set_ylabel("Return")
axes[0].set_title("Quantile functions")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Transport map T(x) = Q₂(F₁(x))
T = optimal_transport_map_1d(mu1, mu2, n_quantiles=500)
x_range = np.linspace(-3, 3, 200)
axes[1].plot(x_range, T(x_range), color="forestgreen", lw=2, label="T(x)")
axes[1].plot(x_range, x_range, "k--", alpha=0.4, lw=1, label="Identity")
axes[1].set_xlabel("x"); axes[1].set_ylabel("T(x)")
axes[1].set_title("Optimal transport map T: μ₁ → μ₂")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/ot_primer_maps.png", dpi=100)
plt.show()
print("Affine map: slope =", (T(1.0) - T(0.0)), "  (theory: σ₂/σ₁ = 1.5)")


Affine map: slope = 1.5250684928624398   (theory: σ₂/σ₁ = 1.5)


## 3. Transport Cost Decomposition

For arbitrary distributions, $W_2^2 = \underbrace{(\bar\mu_1 - \bar\mu_2)^2}_{\text{location}} + \underbrace{(\sigma_1 - \sigma_2)^2}_{\text{scale}} + \underbrace{\text{residual}}_{\text{shape}}$

The shape term measures transport cost attributable to differences in higher-order structure (skewness, tails, multi-modality).


In [4]:
decomp = transport_cost_decomposition(mu1, mu2)
print("Transport cost decomposition:")
for k, v in decomp.items():
    print(f"  {k:20s}: {v:.6f}")


Transport cost decomposition:
  total               : 4.349285
  location            : 4.099644
  scale               : 0.244202
  shape               : 0.005440
  location_fraction   : 0.942602
  scale_fraction      : 0.056148
  shape_fraction      : 0.001251


## 4. Displacement Interpolation (McCann 1997)

The W₂-**geodesic** between $\mu_0$ and $\mu_1$ is:
$$\mu_t = ((1-t)\,\mathrm{Id} + t\,T)_\# \mu_0, \quad t \in [0,1]$$

In 1-d: $F_{\mu_t}^{-1}(u) = (1-t)F_{\mu_0}^{-1}(u) + t F_{\mu_1}^{-1}(u)$

This path has **constant speed**: $W_2(\mu_0, \mu_t) = t \cdot W_2(\mu_0, \mu_1)$.

Compare to Euclidean interpolation of parameters — that path does *not* stay on the geodesic.


In [5]:
path = interpolation_path(mu1, mu2, n_steps=10, n_support=500)
t_vals = np.linspace(0, 1, len(path))

# Verify constant speed
w2_vals = [wasserstein_1d(path[0], p) for p in path]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cmap = plt.cm.coolwarm
for k, (t, dist) in enumerate(zip(t_vals, path)):
    axes[0].plot(u, dist.quantile_function(u), color=cmap(t), lw=1.5, alpha=0.8)
axes[0].set_xlabel("Quantile u"); axes[0].set_ylabel("Return")
axes[0].set_title("W₂-geodesic: quantile functions")
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0,1))
plt.colorbar(sm, ax=axes[0], label="t")
axes[0].grid(alpha=0.3)

axes[1].plot(t_vals, w2_vals, "o-", color="steelblue", label="Actual W₂(μ₀, μₜ)", lw=2)
axes[1].plot(t_vals, [t * w2_vals[-1] for t in t_vals], "r--", lw=1.5,
             label="Expected: t·W₂(μ₀,μ₁)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("W₂ distance")
axes[1].set_title("Constant-speed verification")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/geodesic.png", dpi=100)
plt.show()
print(f"Max deviation from constant speed: {max(abs(w2_vals[k] - t*w2_vals[-1]) for k,t in enumerate(t_vals)):.6f}")


Max deviation from constant speed: 0.000346


## 5. Why Not KL Divergence?

KL divergence $D_{KL}(\mu \| \nu) = \int \log\frac{d\mu}{d\nu} d\mu$ is **not a metric** (asymmetric, undefined when supports differ).  It measures information, not geometry.

Wasserstein distance:
- Is a **true metric** (symmetric, triangle inequality).
- Metrises **weak convergence**: small W₂ ⟺ nearby distributions.
- Measures the minimum **work** to transport mass.
- Gives meaningful distances even between disjoint supports.


In [6]:
# Demonstration: distributions with disjoint supports
a = EmpiricalDistribution(rng.normal(-5, 0.1, 500))
b = EmpiricalDistribution(rng.normal( 5, 0.1, 500))

w2 = wasserstein_1d(a, b, p=2)
print(f"W₂(disjoint supports): {w2:.4f}  (well-defined, = ~10)")
print("KL: undefined (disjoint supports, KL = ∞)")

# W₂ satisfies triangle inequality
c = EmpiricalDistribution(rng.normal(0, 1, 500))
dac = wasserstein_1d(a, c, p=2)
dcb = wasserstein_1d(c, b, p=2)
dab = wasserstein_1d(a, b, p=2)
print(f"\nTriangle inequality: W₂(a,b)={dab:.3f} ≤ W₂(a,c)+W₂(c,b)={dac+dcb:.3f}  ✓")
print("Done.")


W₂(disjoint supports): 10.0017  (well-defined, = ~10)
KL: undefined (disjoint supports, KL = ∞)

Triangle inequality: W₂(a,b)=10.002 ≤ W₂(a,c)+W₂(c,b)=10.167  ✓
Done.
